# AI vs Real Image Detection Pipeline (v2 — All Bugs Fixed)

**Label Convention (Ground Truth) -- Used Consistently Throughout:**
- **Real Image = `0`** (original and all transformations)
- **AI Image = `1`** (original and all transformations)

### Pipeline Overview:
1. Download & preprocess dataset (convert to PNG with EXIF preservation, deduplicate, extract metadata from originals)
2. Apply 38 transformations to both real and AI images
3. Run 6 improved detectors on each (original + transformed) image
4. Train ensemble of 3 models (XGBoost, LightGBM, RandomForest) with soft voting
5. Group-aware train/test split (no data leakage)
6. Cross-validated evaluation + model persistence
7. Store correctly classified original images

### Key Fixes in v2:
- **Metadata detector**: 16 forensic features instead of 4 trivial ones
- **SigLIP/ViT**: Auto-detect AI label index, normalized to Real=0/AI=1, GPU acceleration
- **FFT**: Log-scale, radial band decomposition, size-normalized
- **ELA**: Multi-quality (Q95+Q75), skewness, kurtosis, removed redundancy
- **Noise**: 3 denoising methods, per-channel analysis, 6 features
- **Data leakage fix**: GroupShuffleSplit ensures no image appears in both train and test
- **EXIF preservation**: PNG conversion preserves EXIF metadata
- **Feature cleanup**: Removed attack_type/attack_strength from model features

In [29]:
import os
from pathlib import Path

# 1. INPUT DIRECTORY (Read-only, where the images are)
INPUT_DIR = "/kaggle/input/datasets/ishu15m/ai-vs-real-images"
DATASET = Path(INPUT_DIR)

# 2. OUTPUT DIRECTORY (Writable, where the CSVs will go)
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. SET FILES TO SAVE IN THE OUTPUT DIRECTORY
REAL_CSV = os.path.join(OUTPUT_DIR, "real_detector_dataset_v2.csv")
AI_CSV = os.path.join(OUTPUT_DIR, "ai_detector_dataset_v2.csv")
COMBINED_CSV = os.path.join(OUTPUT_DIR, "combined_detector_dataset_v2.csv")

# Storage folders for correctly classified originals
CORRECT_REAL_DIR = os.path.join(OUTPUT_DIR, "correctly_classified_v2/real")
CORRECT_AI_DIR = os.path.join(OUTPUT_DIR, "correctly_classified_v2/ai")
os.makedirs(CORRECT_REAL_DIR, exist_ok=True)
os.makedirs(CORRECT_AI_DIR, exist_ok=True)

# Model output directory
MODEL_DIR = os.path.join(OUTPUT_DIR, "models_v2")
os.makedirs(MODEL_DIR, exist_ok=True)

# Keep PROJECT_DIR pointing to the output directory so later cells don't crash
PROJECT_DIR = OUTPUT_DIR 


In [30]:
# ==========================================================
# CELL 3 -- Download Primary Dataset from Kaggle
# ==========================================================

import kagglehub

path = kagglehub.dataset_download("ishu15m/ai-vs-real-images")
print("Path to dataset files:", path)

DATASET = Path(path)

# Show folder structure
print("\nDataset structure:")
for item in sorted(DATASET.rglob("*")):
    if item.is_dir():
        count = sum(1 for f in item.iterdir() if f.is_file())
        print(f"  [DIR]  {item.relative_to(DATASET)}  ({count} files)")

Path to dataset files: /kaggle/input/datasets/ishu15m/ai-vs-real-images

Dataset structure:
  [DIR]  AI-images  (0 files)
  [DIR]  AI-images/AI-images  (0 files)
  [DIR]  AI-images/AI-images/ai_animals  (50 files)
  [DIR]  AI-images/AI-images/ai_buildings  (50 files)
  [DIR]  AI-images/AI-images/ai_food  (31 files)
  [DIR]  AI-images/AI-images/ai_human  (35 files)
  [DIR]  AI-images/AI-images/ai_interior  (33 files)
  [DIR]  AI-images/AI-images/ai_items  (34 files)
  [DIR]  AI-images/AI-images/ai_nature  (45 files)
  [DIR]  Real-images  (0 files)
  [DIR]  Real-images/Real-images  (0 files)
  [DIR]  Real-images/Real-images/real_animals  (50 files)
  [DIR]  Real-images/Real-images/real_buildings  (53 files)
  [DIR]  Real-images/Real-images/real_food  (33 files)
  [DIR]  Real-images/Real-images/real_humans  (50 files)
  [DIR]  Real-images/Real-images/real_interior  (28 files)
  [DIR]  Real-images/Real-images/real_items  (34 files)
  [DIR]  Real-images/Real-images/real_nature  (50 files)


### Data Preprocessing
Steps: Extract metadata from ORIGINALS -> Convert to PNG (with EXIF preservation) -> Remove duplicates (perceptual hash) -> Verify counts

**Important**: Metadata is extracted from the original images BEFORE PNG conversion to preserve EXIF data.
Although EXIF features were dropped from training, structural metadata is still extracted.

In [31]:
# ==========================================================
# metadata.py -- Extract Image Metadata from ORIGINALS
# ==========================================================
# Scans primary dataset.

from pathlib import Path
from PIL import Image
import pandas as pd
import os
# Collect all dataset folders to scan
all_dataset_folders = list(DATASET.iterdir())
print("Scanning primary dataset only for metadata.")

meta_rows = []
VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".avif", ".tif", ".tiff"}

for folder in all_dataset_folders:
    if not folder.is_dir():
        continue
    label = folder.name

    for img_path in folder.rglob("*"):
        if img_path.suffix.lower() not in VALID_EXTENSIONS:
            continue

        try:
            with Image.open(img_path) as img:
                width, height = img.size
                img_format = img.format

                meta_rows.append({
                    "image_id": img_path.name,
                    "label": label,
                    "source": "primary",
                    "width": width,
                    "height": height,
                    "format": img_format,
                })
        except Exception as e:
            print(f"Skipped {img_path.name}: {e}")

meta_df = pd.DataFrame(meta_rows)
meta_output = os.path.join(PROJECT_DIR, "image_metadata_v2.csv")
meta_df.to_csv(meta_output, index=False)

print(f"\nSaved metadata for {len(meta_df)} images to {meta_output}")
print(meta_df.head())

Scanning primary dataset only for metadata.

Saved metadata for 576 images to /kaggle/working/image_metadata_v2.csv
    image_id        label   source  width  height format
0   (2).jpeg  Real-images  primary    225     225   JPEG
1  (12).jpeg  Real-images  primary    173     291   JPEG
2  (33).jpeg  Real-images  primary    194     259   JPEG
3  (23).jpeg  Real-images  primary    213     237   JPEG
4  (15).jpeg  Real-images  primary    225     225   JPEG


In [32]:
# ==========================================================
# convert_to_png.py -- Convert All Images to PNG (EXIF preserved)
# ==========================================================

from pathlib import Path
from PIL import Image
from tqdm import tqdm

SOURCE = DATASET
DESTINATION = Path(os.path.join(PROJECT_DIR, "png_dataset"))
DESTINATION.mkdir(parents=True, exist_ok=True)

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".avif"}

all_files = []
for file in SOURCE.rglob("*"):
    if file.suffix.lower() in VALID_EXTENSIONS:
        all_files.append(file)

print(f"Found {len(all_files)} images to convert")

for file in tqdm(all_files, desc="Converting to PNG"):
    relative_path = file.relative_to(SOURCE)
    output_file = DESTINATION / relative_path.with_suffix(".png")
    output_file.parent.mkdir(parents=True, exist_ok=True)

    try:
        img = Image.open(file)
        # Preserve EXIF data during conversion
        exif_data = img.info.get("exif", None)
        img = img.convert("RGB")
        if exif_data:
            img.save(output_file, format="PNG", exif=exif_data)
        else:
            img.save(output_file, format="PNG")
    except Exception as e:
        print(f"Error: {file} -- {e}")

print("Conversion complete (EXIF preserved where available).")

Found 576 images to convert


Converting to PNG: 100%|██████████| 576/576 [07:32<00:00,  1.27it/s]

Conversion complete (EXIF preserved where available).


In [33]:
# ==========================================================
# hash.py -- Remove Duplicate Images Using Perceptual Hashing
# ==========================================================

from pathlib import Path
from PIL import Image
import imagehash

PNG_DATASET = Path(os.path.join(PROJECT_DIR, "png_dataset"))

# Recursively find all folders (not just top-level)
folders = [p for p in PNG_DATASET.iterdir() if p.is_dir()]
extensions = {".png"}

for folder in folders:
    print(f"\n{'='*80}")
    print(f"Processing: {folder.name}")
    print(f"{'='*80}")

    hash_dict = {}
    duplicates_found = 0

    for img_path in folder.rglob("*"):
        if img_path.suffix.lower() not in extensions:
            continue

        try:
            with Image.open(img_path) as img:
                phash = imagehash.phash(img)

            if phash in hash_dict:
                original = hash_dict[phash]
                print(f"\n[DUPLICATE] Kept: {original.name} | Deleted: {img_path.name}")
                img_path.unlink()
                duplicates_found += 1
            else:
                hash_dict[phash] = img_path

        except Exception as e:
            print(f"Error processing {img_path}: {e}")

    print(f"\nTotal duplicates removed: {duplicates_found}")


Processing: Real-images

Total duplicates removed: 0

Processing: AI-images

[DUPLICATE] Kept: (24).png | Deleted: (23).png

[DUPLICATE] Kept: (22).png | Deleted: (18).png

[DUPLICATE] Kept: (22).png | Deleted: (16).png

Total duplicates removed: 3


In [34]:
# ==========================================================
# verify.py + count.py -- Count and Verify Images
# ==========================================================

from pathlib import Path

PNG_DATASET = Path(os.path.join(PROJECT_DIR, "png_dataset"))

print("Image counts by folder:")
print("=" * 50)

for folder in sorted(PNG_DATASET.iterdir()):
    if folder.is_dir():
        count = len(list(folder.rglob("*.png")))
        print(f"  {folder.name:30s} : {count:6d}")

total = len(list(PNG_DATASET.rglob("*.png")))
print(f"{'':30s}   {'---':>6}")
print(f"  {'TOTAL':30s} : {total:6d}")

Image counts by folder:
  AI-images                      :    257
  Real-images                    :    269
                                    ---
  TOTAL                          :    526


### Transformations
38 total transformations including: JPEG compression, blur, sharpness, brightness, contrast, noise, rotation, flip, hue shift, saturation, resize, crop, and screenshot simulations.

Each function takes a PIL Image and returns a PIL Image.

In [35]:
# ==========================================================
# transformations.py -- All Image Transformation Functions
# ==========================================================
# Each function takes a PIL Image and returns a PIL Image.
# "none" transformation returns the image unchanged.

from PIL import Image, ImageEnhance, ImageFilter
from io import BytesIO
import numpy as np
import cv2

# --- JPEG Compression ---
def jpeg_compress(img, quality):
    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    result = Image.open(buffer).convert("RGB")
    result.load()  # Force load before buffer is GC'd
    return result

# --- Gaussian Blur ---
def gaussian_blur(img, radius):
    return img.filter(ImageFilter.GaussianBlur(radius))

# --- Sharpness ---
def sharpen(img, factor):
    return ImageEnhance.Sharpness(img).enhance(factor)

# --- Brightness ---
def brightness(img, factor):
    return ImageEnhance.Brightness(img).enhance(factor)

# --- Contrast ---
def contrast(img, factor):
    return ImageEnhance.Contrast(img).enhance(factor)

# --- Gaussian Noise ---
def gaussian_noise(img, sigma):
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, sigma, arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

# --- Rotation ---
def rotate(img, angle):
    return img.rotate(angle, expand=False, fillcolor=(128, 128, 128))

# --- Horizontal Flip ---
def horizontal_flip(img):
    return img.transpose(Image.FLIP_LEFT_RIGHT)

# --- Hue Shift ---
def hue_shift(img, shift):
    hsv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2HSV)
    hsv[:, :, 0] = (hsv[:, :, 0].astype(int) + shift) % 180
    return Image.fromarray(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB))

# --- Saturation ---
def saturation(img, factor):
    return ImageEnhance.Color(img).enhance(factor)

# --- Resize Scale ---
def resize_scale(img, scale):
    w, h = img.size
    return img.resize((max(1, int(w * scale)), max(1, int(h * scale))))

# --- Center Crop ---
def center_crop(img, percent):
    w, h = img.size
    nw = int(w * percent)
    nh = int(h * percent)
    left = (w - nw) // 2
    top = (h - nh) // 2
    return img.crop((left, top, left + nw, top + nh))

# --- Screenshot Simulations ---
def screenshot_phone(img):
    w, h = img.size
    new_h = int(h * (1080 / w))
    img = img.resize((1080, new_h))
    buffer = BytesIO()
    img.save(buffer, format="PNG")
    buffer.seek(0)
    result = Image.open(buffer).convert("RGB")
    result.load()
    return result

def screenshot_social(img):
    w, h = img.size
    new_h = int(h * (1080 / w))
    img = img.resize((1080, new_h))
    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=85)
    buffer.seek(0)
    result = Image.open(buffer).convert("RGB")
    result.load()
    return result

def screenshot_messaging(img):
    w, h = img.size
    new_h = int(h * (720 / w))
    img = img.resize((720, new_h))
    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=70)
    buffer.seek(0)
    result = Image.open(buffer).convert("RGB")
    result.load()
    return result

# ==========================================================
# COMPLETE TRANSFORMATIONS DICTIONARY (38 entries)
# ==========================================================

transformations = {
    "none":           lambda x: x,

    # JPEG Compression (3 levels)
    "jpeg_90":        lambda x: jpeg_compress(x, 90),
    "jpeg_70":        lambda x: jpeg_compress(x, 70),
    "jpeg_50":        lambda x: jpeg_compress(x, 50),

    # Gaussian Blur (3 levels)
    "blur_2":         lambda x: gaussian_blur(x, 2),
    "blur_4":         lambda x: gaussian_blur(x, 4),
    "blur_6":         lambda x: gaussian_blur(x, 6),

    # Sharpness (3 levels)
    "sharp_1.5":      lambda x: sharpen(x, 1.5),
    "sharp_2":        lambda x: sharpen(x, 2),
    "sharp_3":        lambda x: sharpen(x, 3),

    # Brightness (3 levels)
    "bright_0.7":     lambda x: brightness(x, 0.7),
    "bright_1.3":     lambda x: brightness(x, 1.3),
    "bright_1.6":     lambda x: brightness(x, 1.6),

    # Contrast (3 levels)
    "contrast_0.7":   lambda x: contrast(x, 0.7),
    "contrast_1.3":   lambda x: contrast(x, 1.3),
    "contrast_1.6":   lambda x: contrast(x, 1.6),

    # Gaussian Noise (3 levels)
    "noise_5":        lambda x: gaussian_noise(x, 5),
    "noise_15":       lambda x: gaussian_noise(x, 15),
    "noise_30":       lambda x: gaussian_noise(x, 30),

    # Rotation (3 levels)
    "rotate_5":       lambda x: rotate(x, 5),
    "rotate_15":      lambda x: rotate(x, 15),
    "rotate_30":      lambda x: rotate(x, 30),

    # Flip (1 level)
    "flip":           lambda x: horizontal_flip(x),

    # Hue Shift (3 levels)
    "hue_10":         lambda x: hue_shift(x, 10),
    "hue_30":         lambda x: hue_shift(x, 30),
    "hue_60":         lambda x: hue_shift(x, 60),

    # Saturation (3 levels)
    "sat_0.7":        lambda x: saturation(x, 0.7),
    "sat_1.3":        lambda x: saturation(x, 1.3),
    "sat_1.8":        lambda x: saturation(x, 1.8),

    # Resize (3 levels)
    "resize_75":      lambda x: resize_scale(x, 0.75),
    "resize_50":      lambda x: resize_scale(x, 0.50),
    "resize_25":      lambda x: resize_scale(x, 0.25),

    # Center Crop (3 levels)
    "crop_95":        lambda x: center_crop(x, 0.95),
    "crop_85":        lambda x: center_crop(x, 0.85),
    "crop_70":        lambda x: center_crop(x, 0.70),

    # Screenshot Simulations (3 levels)
    "screenshot_phone":     lambda x: screenshot_phone(x),
    "screenshot_social":    lambda x: screenshot_social(x),
    "screenshot_messaging": lambda x: screenshot_messaging(x),
}

print(f"{len(transformations)} transformations loaded.")

38 transformations loaded.


### Detectors (Improved v2)
6 detectors with improved feature extraction:
1. **SigLIP** (HuggingFace) -- Ateeqq/ai-vs-human-image-detector — **label-normalized, GPU**
2. **ViT** (HuggingFace) -- dima806/ai_vs_human_generated_image_detection — **label-normalized, GPU**
3. **FFT** -- Frequency domain analysis — **radial bands, log-scale, size-normalized**
4. **ELA** -- Error Level Analysis — **multi-quality (Q95+Q75), skew, kurtosis**
5. **Noise** -- Noise residual analysis — **3 methods, per-channel, 6 features**
6. **Metadata** -- Forensic metadata check — **16 features instead of 4**

In [36]:
# ==========================================================
# detector_1.py -- SigLIP AI vs Human Image Detector (BATCHED)
# ==========================================================

from transformers import AutoImageProcessor, AutoModelForImageClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

siglip_model_name = "Ateeqq/ai-vs-human-image-detector"
siglip_processor = AutoImageProcessor.from_pretrained(siglip_model_name)
siglip_model = AutoModelForImageClassification.from_pretrained(siglip_model_name).to(device)

_siglip_id2label = siglip_model.config.id2label
_siglip_ai_idx = None
for idx, label in _siglip_id2label.items():
    if any(kw in str(label).lower() for kw in ["ai", "fake", "generated", "artificial"]):
        _siglip_ai_idx = int(idx)
        break
if _siglip_ai_idx is None:
    _siglip_ai_idx = 1
    print(f"  [WARNING] Could not auto-detect AI index, defaulting to {_siglip_ai_idx}")
else:
    print(f"  SigLIP AI class index = {_siglip_ai_idx}")

def siglip_batch_detector(images_list):
    if not images_list: return []
    inputs = siglip_processor(images=images_list, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = siglip_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    
    results = []
    for i in range(len(images_list)):
        ai_prob = float(probs[i, _siglip_ai_idx])
        results.append({
            "siglip_ai_prob": ai_prob,
            "siglip_confidence": float(probs[i].max()),
        })
    return results

print("SigLIP batch detector loaded (label-normalized, GPU batching).")


Using device: cuda


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

  SigLIP AI class index = 0
SigLIP batch detector loaded (label-normalized, GPU batching).


In [37]:
# ==========================================================
# detector_2.py -- ViT AI vs Human Image Detector (BATCHED)
# ==========================================================

from transformers import AutoImageProcessor, AutoModelForImageClassification
import torch

vit_model_name = "dima806/ai_vs_human_generated_image_detection"
vit_processor = AutoImageProcessor.from_pretrained(vit_model_name)
vit_model = AutoModelForImageClassification.from_pretrained(vit_model_name).to(device)

_vit_id2label = vit_model.config.id2label
_vit_ai_idx = None
for idx, label in _vit_id2label.items():
    if any(kw in str(label).lower() for kw in ["ai", "fake", "generated", "artificial"]):
        _vit_ai_idx = int(idx)
        break
if _vit_ai_idx is None:
    _vit_ai_idx = 1
    print(f"  [WARNING] Could not auto-detect AI index, defaulting to {_vit_ai_idx}")
else:
    print(f"  ViT AI class index = {_vit_ai_idx}")

def vit_batch_detector(images_list):
    if not images_list: return []
    inputs = vit_processor(images=images_list, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = vit_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    
    results = []
    for i in range(len(images_list)):
        ai_prob = float(probs[i, _vit_ai_idx])
        results.append({
            "vit_ai_prob": ai_prob,
            "vit_confidence": float(probs[i].max()),
        })
    return results

print("ViT batch detector loaded (label-normalized, GPU batching).")


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  ViT AI class index = 1
ViT batch detector loaded (label-normalized, GPU batching).


In [38]:
# ==========================================================
# detector_3.py -- FFT Frequency Analysis Detector (IMPROVED)
# ==========================================================
# FIX: Log-scale magnitudes for stability
# FIX: Radial band decomposition (low/mid/high)
# FIX: Size-normalized features
# NEW: mid-to-high ratio feature

import numpy as np
import cv2

def fft_detector(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY).astype(np.float32)
    h, w = gray.shape

    fft = np.fft.fft2(gray)
    fft_shift = np.fft.fftshift(fft)
    magnitude = np.log1p(np.abs(fft_shift))  # Log-scale

    # Radial frequency bands
    cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w]
    r = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2)
    r_max = np.sqrt(cx ** 2 + cy ** 2) + 1e-10
    r_norm = r / r_max

    low_mask  = r_norm <= 0.33
    mid_mask  = (r_norm > 0.33) & (r_norm <= 0.66)
    high_mask = r_norm > 0.66

    low_energy  = float(np.mean(magnitude[low_mask]))  if low_mask.any()  else 0.0
    mid_energy  = float(np.mean(magnitude[mid_mask]))  if mid_mask.any()  else 0.0
    high_energy = float(np.mean(magnitude[high_mask])) if high_mask.any() else 0.0

    total_energy = float(np.sum(magnitude)) + 1e-10
    high_freq_ratio = float(np.sum(magnitude[high_mask]) / total_energy)

    # Spectral entropy
    mag_flat = magnitude.flatten()
    mag_prob = mag_flat / (mag_flat.sum() + 1e-10)
    spectral_entropy = float(-np.sum(mag_prob * np.log(mag_prob + 1e-10)))

    return {
        "fft_low_energy":        low_energy,
        "fft_mid_energy":        mid_energy,
        "fft_high_energy":       high_energy,
        "fft_high_freq_ratio":   high_freq_ratio,
        "fft_entropy":           spectral_entropy,
        "fft_mid_to_high_ratio": float(mid_energy / (high_energy + 1e-10)),
    }

print("FFT detector loaded (radial bands, log-scale, 6 features).")

FFT detector loaded (radial bands, log-scale, 6 features).


In [39]:
# ==========================================================
# detector_4.py -- Error Level Analysis (ELA) Detector (FIXED)
# ==========================================================
# FIX: Image.open(buffer) instead of image.open(buffer)
# FIX: Removed redundant ela_score (was identical to ela_mean)
# NEW: Multi-quality ELA (Q95 + Q75)
# NEW: Skewness and kurtosis features

from io import BytesIO
from PIL import Image, ImageChops
import numpy as np

def ela_detector(image):
    img_rgb = image.convert("RGB")

    # ELA at quality 95
    buffer95 = BytesIO()
    img_rgb.save(buffer95, format="JPEG", quality=95)
    buffer95.seek(0)
    recomp95 = Image.open(buffer95).convert("RGB")
    ela95 = np.array(ImageChops.difference(img_rgb, recomp95)).astype(np.float32)

    # ELA at quality 75 (reveals different artifacts)
    buffer75 = BytesIO()
    img_rgb.save(buffer75, format="JPEG", quality=75)
    buffer75.seek(0)
    recomp75 = Image.open(buffer75).convert("RGB")
    ela75 = np.array(ImageChops.difference(img_rgb, recomp75)).astype(np.float32)

    # Statistical features
    ela95_std = float(np.std(ela95)) + 1e-8

    return {
        "ela_mean_q95":  float(np.mean(ela95)),
        "ela_std_q95":   float(np.std(ela95)),
        "ela_max_q95":   float(np.max(ela95)),
        "ela_mean_q75":  float(np.mean(ela75)),
        "ela_std_q75":   float(np.std(ela75)),
        "ela_skew":      float(np.mean(((ela95 - np.mean(ela95)) / ela95_std) ** 3)),
        "ela_kurtosis":  float(np.mean(((ela95 - np.mean(ela95)) / ela95_std) ** 4)),
    }

print("ELA detector loaded (multi-quality, 7 features).")

ELA detector loaded (multi-quality, 7 features).


In [40]:
# ==========================================================
# detector_5.py -- Noise Residual Detector (IMPROVED)
# ==========================================================
# NEW: 3 denoising methods (Gaussian, Median, Laplacian)
# NEW: Per-channel noise analysis
# NEW: 6 features instead of 1

import numpy as np
import cv2

def noise_detector(image):
    img = np.array(image).astype(np.float32)

    # Method 1: Gaussian blur residual
    denoised_gauss = cv2.GaussianBlur(img, (5, 5), 0)
    residual_gauss = img - denoised_gauss

    # Method 2: Median filter residual
    gray = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)
    denoised_median = cv2.medianBlur(gray.astype(np.uint8), 5).astype(np.float32)
    residual_median = gray - denoised_median

    # Method 3: Laplacian (high-frequency noise)
    laplacian = cv2.Laplacian(gray, cv2.CV_32F)

    # Per-channel noise analysis
    n_channels = img.shape[2] if img.ndim == 3 else 1
    channel_stds = [float(np.std(residual_gauss[:, :, c])) for c in range(min(3, n_channels))]
    if len(channel_stds) < 3:
        channel_stds = channel_stds + [0.0] * (3 - len(channel_stds))

    return {
        "noise_std_gauss":         float(np.std(residual_gauss)),
        "noise_mean_gauss":        float(np.mean(np.abs(residual_gauss))),
        "noise_std_median":        float(np.std(residual_median)),
        "noise_laplacian_var":     float(np.var(laplacian)),
        "noise_channel_std_range": float(max(channel_stds) - min(channel_stds)),
        "noise_channel_std_mean":  float(np.mean(channel_stds)),
    }

print("Noise detector loaded (3 methods, 6 features).")

Noise detector loaded (3 methods, 6 features).


In [41]:
# ==========================================================
# detector_6.py -- Forensic Metadata Detector (REWRITTEN)
# ==========================================================
# REWRITE: 16 forensic features instead of 4 trivial ones.
# Includes: EXIF field counting, camera metadata presence,
#           AI-typical dimension detection, ICC profile check,
#           quantization table detection, and more.

def metadata_detector(image):
    import math
    # Image properties
    w, h = image.size
    aspect_ratio = w / max(h, 1)
    total_pixels = w * h

    # AI-typical dimensions
    is_square = (w == h)
    is_common_ai_size = (w, h) in [
        (512, 512), (768, 768), (1024, 1024), (256, 256),
        (512, 768), (768, 512), (1024, 768), (768, 1024),
    ]
    is_power_of_2 = (w > 0 and h > 0 and (w & (w - 1) == 0) and (h & (h - 1) == 0))

    # Channel analysis
    mode = image.mode
    num_channels = len(image.getbands())
    has_alpha = int("A" in mode or mode == "RGBA")

    # JPEG quantization table
    has_quantization = int(hasattr(image, 'quantization') and image.quantization is not None)

    # ICC color profile
    has_icc_profile = int(image.info.get("icc_profile") is not None)

    return {
        "meta_has_icc_profile":    has_icc_profile,
        "meta_aspect_ratio":       float(aspect_ratio),
        "meta_total_pixels":       float(math.log1p(total_pixels)),
        "meta_is_square":          int(is_square),
        "meta_is_common_ai_size":  int(is_common_ai_size),
        "meta_is_power_of_2":      int(is_power_of_2),
        "meta_has_alpha":          has_alpha,
        "meta_num_channels":       int(num_channels),
    }

print("Metadata detector loaded (9 structural features).")

Metadata detector loaded (9 structural features).


In [42]:
# ==========================================================
# all_detectors.py -- CPU Detectors helper (BATCHED)
# ==========================================================

def cpu_detectors(image):
    result = {}
    result.update(fft_detector(image))
    result.update(ela_detector(image))
    result.update(noise_detector(image))
    result.update(metadata_detector(image))
    return result

import concurrent.futures

def run_all_detectors_batched(original_img, transformations_dict):
    attack_names = list(transformations_dict.keys())
    
    # 1. Generate all transformations
    transformed_images = []
    for attack_name in attack_names:
        transformed = transformations_dict[attack_name](original_img.copy())
        transformed_images.append(transformed)
        
    # 2. Batch GPU Processing
    siglip_results = siglip_batch_detector(transformed_images)
    vit_results = vit_batch_detector(transformed_images)
    
    # 3. Parallel CPU Processing
    cpu_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        cpu_results = list(executor.map(cpu_detectors, transformed_images))
        
    # 4. Combine
    final_scores = []
    for i in range(len(attack_names)):
        scores = {}
        scores.update(siglip_results[i])
        scores.update(vit_results[i])
        scores.update(cpu_results[i])
        final_scores.append(scores)
        
    return attack_names, final_scores

print("Batched pipeline helper loaded.")


Batched pipeline helper loaded.


### Process Real Images (Ground Truth Label = 0)
Apply all 38 transformations to each real image, run all 6 detectors, save to CSV.

**Convention: Every row from a real image (original or transformed) gets label = 0.**

In [43]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
2
Tesla T4


### Process AI Images (Ground Truth Label = 1)
Same pipeline as real images but with label = 1.

**Convention: Every row from an AI image (original or transformed) gets label = 1.**

In [44]:
# ==========================================================
# Process REAL Images -- Label = 0 (BATCHED)
# ==========================================================

from pathlib import Path
from PIL import Image
from tqdm import tqdm
import pandas as pd

REAL_FOLDER = DATASET / "Real-images" / "Real-images"
if not REAL_FOLDER.exists(): REAL_FOLDER = DATASET / "Real-images"
if not REAL_FOLDER.exists():
    for p in sorted(DATASET.rglob("*")):
        if p.is_dir() and "real" in p.name.lower():
            REAL_FOLDER = p
            break
print(f"Real images folder: {REAL_FOLDER}")
assert REAL_FOLDER.exists(), f"Real folder not found: {REAL_FOLDER}"

processed_ids = set()
if Path(REAL_CSV).exists():
    old_df = pd.read_csv(REAL_CSV)
    processed_ids = set(old_df["image_id"].astype(str).tolist())
    print(f"Resuming from {len(processed_ids)} samples")

image_files = []
for ext in ["*.png", "*.jpg", "*.jpeg", "*.webp", "*.avif"]:
    for f in REAL_FOLDER.rglob(ext): image_files.append(f)

print(f"Found {len(image_files)} real images")

rows = []
for image_path in tqdm(image_files, desc="Real Images"):
    try:
        original = Image.open(image_path).convert("RGB")
        original.thumbnail((1024, 1024))
    except Exception as e:
        tqdm.write(f"FAILED OPENING: {image_path} -- {e}")
        continue
        
    attack_names, batch_scores = run_all_detectors_batched(original, transformations)
    
    for i, attack_name in enumerate(attack_names):
        image_id = str(image_path.relative_to(REAL_FOLDER)).replace("/", "_").replace("\\", "_") + "_" + attack_name
        if image_id in processed_ids: continue
            
        if attack_name == "none":
            attack_type, attack_strength = "none", "0"
        else:
            parts = attack_name.split("_")
            attack_type = parts[0]
            attack_strength = "_".join(parts[1:])
            
        row = {
            "image_id": image_id,
            "original_path": str(image_path),
            "label": 0,
            "attack_type": attack_type,
            "attack_strength": attack_strength,
        }
        row.update(batch_scores[i])
        rows.append(row)
        processed_ids.add(image_id)
        
    if len(rows) >= SAVE_INTERVAL:
        temp_df = pd.DataFrame(rows)
        if Path(REAL_CSV).exists(): temp_df.to_csv(REAL_CSV, mode="a", header=False, index=False)
        else: temp_df.to_csv(REAL_CSV, index=False)
        tqdm.write(f"Checkpoint saved ({len(rows)} rows)")
        rows = []

if rows:
    temp_df = pd.DataFrame(rows)
    
    if Path(REAL_CSV).exists(): temp_df.to_csv(REAL_CSV, mode="a", header=False, index=False)
    else: temp_df.to_csv(REAL_CSV, index=False)

real_df = pd.read_csv(REAL_CSV)
print(f"\nDONE -- Real Images (Shape: {real_df.shape})")


Real images folder: /kaggle/input/datasets/ishu15m/ai-vs-real-images/Real-images/Real-images
Found 298 real images


Real Images:   1%|          | 3/298 [00:19<28:48,  5.86s/it]

Checkpoint saved (114 rows)


Real Images:   2%|▏         | 6/298 [00:27<16:11,  3.33s/it]

Checkpoint saved (114 rows)


...

Checkpoint saved (114 rows)


Real Images: 100%|█████████▉| 297/298 [34:10<00:07,  7.61s/it]

Checkpoint saved (114 rows)


Real Images: 100%|██████████| 298/298 [34:14<00:00,  6.90s/it]


DONE -- Real Images (Shape: (11324, 36))


In [45]:
# ==========================================================
# Process AI Images -- Label = 1 (BATCHED)
# ==========================================================

from pathlib import Path
from PIL import Image
from tqdm import tqdm
import pandas as pd

AI_FOLDER = DATASET / "AI-images" / "AI-images"
if not AI_FOLDER.exists(): AI_FOLDER = DATASET / "AI-images"
if not AI_FOLDER.exists():
    for p in sorted(DATASET.rglob("*")):
        if p.is_dir() and "ai" in p.name.lower():
            AI_FOLDER = p
            break
print(f"AI images folder: {AI_FOLDER}")
assert AI_FOLDER.exists(), f"AI folder not found: {AI_FOLDER}"

ai_processed_ids = set()
if Path(AI_CSV).exists():
    old_df = pd.read_csv(AI_CSV)
    ai_processed_ids = set(old_df["image_id"].astype(str).tolist())
    print(f"Resuming from {len(ai_processed_ids)} samples")

SUPPORTED_EXTS = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".avif")
ai_image_files = [f for f in AI_FOLDER.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS]

print(f"Found {len(ai_image_files)} AI images")

ai_rows = []
for image_path in tqdm(ai_image_files, desc="AI Images"):
    try:
        original = Image.open(image_path).convert("RGB")
        original.thumbnail((1024, 1024))
    except Exception as e:
        tqdm.write(f"FAILED OPENING: {image_path} -- {e}")
        continue
        
    attack_names, batch_scores = run_all_detectors_batched(original, transformations)
    
    for i, attack_name in enumerate(attack_names):
        image_id = str(image_path.relative_to(AI_FOLDER)).replace("/", "_").replace("\\", "_") + "_" + attack_name
        if image_id in ai_processed_ids: continue
            
        if attack_name == "none":
            attack_type, attack_strength = "none", "0"
        else:
            parts = attack_name.split("_")
            attack_type = parts[0]
            attack_strength = "_".join(parts[1:])
            
        row = {
            "image_id": image_id,
            "original_path": str(image_path),
            "label": 1,
            "attack_type": attack_type,
            "attack_strength": attack_strength,
        }
        row.update(batch_scores[i])
        ai_rows.append(row)
        ai_processed_ids.add(image_id)
        
    if len(ai_rows) >= SAVE_INTERVAL:
        temp_df = pd.DataFrame(ai_rows)
        if Path(AI_CSV).exists(): temp_df.to_csv(AI_CSV, mode="a", header=False, index=False)
        else: temp_df.to_csv(AI_CSV, index=False)
        tqdm.write(f"Checkpoint saved ({len(ai_rows)} rows)")
        ai_rows = []

if ai_rows:
    temp_df = pd.DataFrame(ai_rows)
    if Path(AI_CSV).exists(): temp_df.to_csv(AI_CSV, mode="a", header=False, index=False)
    else: temp_df.to_csv(AI_CSV, index=False)

ai_df = pd.read_csv(AI_CSV)
print(f"\nDONE -- AI Images (Shape: {ai_df.shape})")


AI images folder: /kaggle/input/datasets/ishu15m/ai-vs-real-images/AI-images/AI-images
Found 278 AI images


AI Images:   1%|          | 3/278 [00:23<35:30,  7.75s/it]

Checkpoint saved (114 rows)


AI Images:   2%|▏         | 6/278 [00:46<34:55,  7.70s/it]

Checkpoint saved (114 rows)


...

Checkpoint saved (114 rows)


AI Images:  99%|█████████▉| 276/278 [41:01<00:14,  7.45s/it]

Checkpoint saved (114 rows)


AI Images: 100%|██████████| 278/278 [41:16<00:00,  8.91s/it]


DONE -- AI Images (Shape: (10564, 36))


### Ensemble Training (Improved v2)
**3 Models**: XGBoost, LightGBM, RandomForest

**Method**: Soft Voting Ensemble (VotingClassifier)

**Labels**: Real = 0, AI = 1

**Key Fixes**:
- Group-aware train/test split (no data leakage from transformed images)
- attack_type / attack_strength removed from features (not available at inference)
- Cross-validation with GroupKFold
- Model persistence with joblib
- Feature importance analysis

In [46]:
# ==========================================================
# Ensemble Training -- XGBoost + LightGBM + RandomForest (FIXED)
# ==========================================================
# Convention: Real = 0, AI = 1
# FIX: GroupShuffleSplit to prevent data leakage
# FIX: Removed attack_type/attack_strength from features
# NEW: Cross-validation, feature importance, model persistence

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ==========================================================
# LOAD & COMBINE CSVs
# ==========================================================

real_df = pd.read_csv(REAL_CSV)
ai_df = pd.read_csv(AI_CSV)

df = pd.concat([real_df, ai_df], ignore_index=True)

# Save combined CSV
df.to_csv(COMBINED_CSV, index=False)

print(f"Combined shape: {df.shape}")
print(f"Real (label=0): {(df['label'] == 0).sum()}")
print(f"AI   (label=1): {(df['label'] == 1).sum()}")

# ==========================================================
# CLEAN DATA
# ==========================================================

# Drop fully empty columns
df = df.dropna(axis=1, how="all")

# Fill missing values
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("unknown")
    else:
        df[col] = df[col].fillna(df[col].median())

# ==========================================================
# LABEL VERIFICATION -- Real = 0, AI = 1
# ==========================================================

df["label"] = df["label"].astype(int)
assert set(df["label"].unique()) <= {0, 1}, f"Unexpected labels: {df['label'].unique()}"

print(f"\nLabel distribution:\n{df['label'].value_counts()}")

# ==========================================================
# FEATURES -- detector outputs only (NO attack_type/strength)
# ==========================================================
# attack_type and attack_strength are NOT available at inference
# time on unseen images, so they must NOT be features.

detector_features = [
    # SigLIP detector (2 features)
    "siglip_ai_prob", "siglip_confidence",
    # ViT detector (2 features)
    "vit_ai_prob", "vit_confidence",
    # FFT detector (6 features)
    "fft_low_energy", "fft_mid_energy", "fft_high_energy",
    "fft_high_freq_ratio", "fft_entropy", "fft_mid_to_high_ratio",
    # ELA detector (7 features)
    "ela_mean_q95", "ela_std_q95", "ela_max_q95",
    "ela_mean_q75", "ela_std_q75", "ela_skew", "ela_kurtosis",
    # Noise detector (6 features)
    "noise_std_gauss", "noise_mean_gauss", "noise_std_median",
    "noise_laplacian_var", "noise_channel_std_range", "noise_channel_std_mean",
    # Metadata detector (8 structural features)
    "meta_has_icc_profile", "meta_aspect_ratio", "meta_total_pixels",
    "meta_is_square", "meta_is_common_ai_size", "meta_is_power_of_2",
    "meta_has_alpha", "meta_num_channels",
]

# Keep only columns that exist in the data
features = [c for c in detector_features if c in df.columns]
missing = [c for c in detector_features if c not in df.columns]
if missing:
    print(f"\n[WARNING] Missing features (not in CSV): {missing}")
print(f"\nFeatures ({len(features)}): {features}")

# ==========================================================
# GROUP-AWARE TRAIN / TEST SPLIT (no data leakage)
# ==========================================================
# Use original_path as the group key so all transformations
# of the same image stay in the same split.

X = df[features]
y = df["label"]
groups = df["original_path"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
id_train = df["image_id"].iloc[train_idx]
id_test = df["image_id"].iloc[test_idx]

print(f"\nTrain: {X_train.shape}  (from {groups.iloc[train_idx].nunique()} unique images)")
print(f"Test : {X_test.shape}  (from {groups.iloc[test_idx].nunique()} unique images)")

# Verify no overlap
train_images = set(groups.iloc[train_idx])
test_images = set(groups.iloc[test_idx])
assert len(train_images & test_images) == 0, "DATA LEAKAGE: images appear in both train and test!"
print("No data leakage detected.")

# ==========================================================
# MODEL 1: XGBOOST
# ==========================================================

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    nthread=1
)

# ==========================================================
# MODEL 2: LIGHTGBM
# ==========================================================

lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42,
    verbose=-1,
    n_jobs=1
)

# ==========================================================
# MODEL 3: RANDOM FOREST
# ==========================================================

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# ==========================================================
# ENSEMBLE (SOFT VOTING)
# ==========================================================

ensemble = VotingClassifier(
    estimators=[
        ("xgb", xgb),
        ("lgbm", lgbm),
        ("rf", rf),
    ],
    voting="soft",
    n_jobs=-1
)

# ==========================================================
# TRAIN
# ==========================================================

print("\nTraining Ensemble (XGBoost + LightGBM + RandomForest)...")
ensemble.fit(X_train, y_train)
print("Training complete.")

# ==========================================================
# PREDICT
# ==========================================================

y_pred = ensemble.predict(X_test)
y_prob = ensemble.predict_proba(X_test)[:, 1]

# ==========================================================
# METRICS
# ==========================================================

print("\n" + "=" * 50)
print("ENSEMBLE RESULTS (Group-aware split, no leakage)")
print("=" * 50)

print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Real (0)", "AI (1)"]))

# ==========================================================
# CROSS-VALIDATION (Group-aware)
# ==========================================================

print("\nRunning 5-fold Group Cross-Validation...")
gkf = GroupKFold(n_splits=5)

# Use a lighter model for CV speed
cv_model = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    eval_metric="logloss", random_state=42
)
cv_scores = cross_val_score(cv_model, X_train, y_train, cv=gkf, groups=groups.iloc[train_idx], scoring="f1")
print(f"CV F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"CV F1 per fold: {[f'{s:.4f}' for s in cv_scores]}")

# ==========================================================
# FEATURE IMPORTANCE
# ==========================================================

print("\n" + "=" * 50)
print("FEATURE IMPORTANCE (from RandomForest)")
print("=" * 50)

rf_fitted = ensemble.named_estimators_["rf"]
importances = pd.Series(rf_fitted.feature_importances_, index=features)
importances = importances.sort_values(ascending=False)
print(importances.to_string())

# ==========================================================
# SAVE MODELS
# ==========================================================

model_path = os.path.join(MODEL_DIR, "ensemble_v2.joblib")
joblib.dump(ensemble, model_path)
print(f"\nModel saved to: {model_path}")

# Save feature list for inference
feature_path = os.path.join(MODEL_DIR, "features_v2.json")
import json
with open(feature_path, "w") as f:
    json.dump(features, f)
print(f"Feature list saved to: {feature_path}")

# ==========================================================
# FALSE POSITIVES / FALSE NEGATIVES
# ==========================================================

results = pd.DataFrame({
    "image_id":       id_test.values,
    "original_path":  df["original_path"].iloc[test_idx].values,
    "actual":         y_test.values,
    "predicted":      y_pred,
    "probability_ai": y_prob
})


# False Positive: Real (0) predicted as AI (1)
fp = results[(results["actual"] == 0) & (results["predicted"] == 1)].copy()
fp["error_type"] = "False Positive"

# False Negative: AI (1) predicted as Real (0)
fn = results[(results["actual"] == 1) & (results["predicted"] == 0)].copy()
fn["error_type"] = "False Negative"

errors = pd.concat([fp, fn], ignore_index=True)
errors.to_csv(os.path.join(PROJECT_DIR, "fp_fn_images_v2.csv"), index=False)

print(f"\nTotal Errors   : {len(errors)}")
print(f"False Positives: {len(fp)}  (Real predicted as AI)")
print(f"False Negatives: {len(fn)}  (AI predicted as Real)")

print("\nSample Errors:")
display(errors.head(20))

Combined shape: (21888, 36)
Real (label=0): 11324
AI   (label=1): 10564

Label distribution:
label
0    11324
1    10564
Name: count, dtype: int64

Features (31): ['siglip_ai_prob', 'siglip_confidence', 'vit_ai_prob', 'vit_confidence', 'fft_low_energy', 'fft_mid_energy', 'fft_high_energy', 'fft_high_freq_ratio', 'fft_entropy', 'fft_mid_to_high_ratio', 'ela_mean_q95', 'ela_std_q95', 'ela_max_q95', 'ela_mean_q75', 'ela_std_q75', 'ela_skew', 'ela_kurtosis', 'noise_std_gauss', 'noise_mean_gauss', 'noise_std_median', 'noise_laplacian_var', 'noise_channel_std_range', 'noise_channel_std_mean', 'meta_has_icc_profile', 'meta_aspect_ratio', 'meta_total_pixels', 'meta_is_square', 'meta_is_common_ai_size', 'meta_is_power_of_2', 'meta_has_alpha', 'meta_num_channels']

Train: (17480, 31)  (from 460 unique images)
Test : (4408, 31)  (from 116 unique images)
No data leakage detected.

Training Ensemble (XGBoost + LightGBM + RandomForest)...
Training complete.

ENSEMBLE RESULTS (Group-aware split, no l

,image_id,original_path,actual,predicted,probability_ai,error_type
0,real_animals_(21).jpg_none,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.867860,False Positive
1,real_animals_(21).jpg_jpeg_90,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.852109,False Positive
2,real_animals_(21).jpg_jpeg_70,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.828329,False Positive
3,real_animals_(21).jpg_jpeg_50,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.784328,False Positive
4,real_animals_(21).jpg_blur_2,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.857202,False Positive
5,real_animals_(21).jpg_blur_4,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.896617,False Positive
6,real_animals_(21).jpg_blur_6,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.898719,False Positive
7,real_animals_(21).jpg_sharp_1.5,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.897607,False Positive
8,real_animals_(21).jpg_sharp_2,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.886471,False Positive
9,real_animals_(21).jpg_sharp_3,/kaggle/input/datasets/ishu15m/ai-vs-real-imag...,0,1,0.826860,False Positive


### Store Correctly Classified Original Images
**Rule:**
- If real image + final vote = 0 --> store original in `correctly_classified_v2/real/`
- If AI image + final vote = 1 --> store original in `correctly_classified_v2/ai/`
- **Only original (untransformed) images are stored**

In [47]:
# ==========================================================
# Store Correctly Classified Original Images
# ==========================================================

import shutil

# --- Map image_id to original info ---
full_df = pd.read_csv(COMBINED_CSV)

id_to_attack = dict(zip(full_df["image_id"], full_df["attack_type"]))
id_to_path = dict(zip(full_df["image_id"], full_df["original_path"]))

# Add original attack type string and path to results
results["attack_type_str"] = results["image_id"].map(id_to_attack)
results["original_path"] = results["image_id"].map(id_to_path)

# ==========================================================
# SECONDARY VERIFICATION PIPELINE (TIER 2 FORENSICS)
# ==========================================================
import cv2
import numpy as np
from PIL import Image

class Tier2Forensics:
    @staticmethod
    def analyze_prnu(image_path):
        try:
            img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            if img is None: return 0.0
            denoised = cv2.fastNlMeansDenoising(img, None, h=10, templateWindowSize=7, searchWindowSize=21)
            residual = img.astype(np.float32) - denoised.astype(np.float32)
            variance = np.var(residual)
            if variance < 1.0: return 0.1
            elif variance > 100.0: return 0.3
            else: return 0.8
        except: return 0.0

    @staticmethod
    def analyze_cfa_bayer(image_path):
        try:
            img = cv2.imread(image_path)
            if img is None: return 0.0
            b, g, r = cv2.split(img)
            f_transform = np.fft.fft2(g)
            f_shift = np.fft.fftshift(f_transform)
            magnitude_spectrum = np.log(np.abs(f_shift) + 1)
            h, w = magnitude_spectrum.shape
            edge_h = np.mean(magnitude_spectrum[:, 0:5]) + np.mean(magnitude_spectrum[:, w-5:w])
            edge_v = np.mean(magnitude_spectrum[0:5, :]) + np.mean(magnitude_spectrum[h-5:h, :])
            corner = np.mean(magnitude_spectrum[0:5, 0:5]) + np.mean(magnitude_spectrum[h-5:h, w-5:w])
            center = np.mean(magnitude_spectrum[h//2-10:h//2+10, w//2-10:w//2+10])
            hf_ratio = (edge_h + edge_v + corner) / (center + 1e-5)
            return float(min(1.0, hf_ratio * 2.0))
        except: return 0.0

    @staticmethod
    def analyze_jpeg_history(image_path):
        try:
            with Image.open(image_path) as img:
                if img.format != 'JPEG': return 0.1 
                qtables = img.quantization if hasattr(img, 'quantization') else None
                if not qtables: return 0.2
                lum_qtable = qtables.get(0, [])
                if len(lum_qtable) < 64: return 0.3
                hf_q_variance = np.var(lum_qtable[32:])
                if hf_q_variance < 10: return 0.4
                else: return 0.9
        except: return 0.0

    @staticmethod
    def run_all_tier_2(image_path):
        prnu = Tier2Forensics.analyze_prnu(image_path)
        cfa = Tier2Forensics.analyze_cfa_bayer(image_path)
        jpeg = Tier2Forensics.analyze_jpeg_history(image_path)
        return (prnu * 0.4) + (cfa * 0.4) + (jpeg * 0.2)

# Find all False Positives from Tier 1 (Actual = 0, Predicted = 1)
false_positives = results[(results["actual"] == 0) & (results["predicted"] == 1)]
print(f"\n[Secondary Pipeline] Found {len(false_positives)} False Positives. Routing to Tier 2 Deep Forensics...")

fixed_count = 0
# Route each false positive through the forensics checks
for idx, row in false_positives.iterrows():
    img_path = str(row["original_path"])
    if not os.path.exists(img_path): continue
    
    # Only run forensics on borderline/medium confidence false positives
    if 0.10 < row["probability_ai"] < 0.95: 
        camera_score = Tier2Forensics.run_all_tier_2(img_path)
        
        # If strong physical camera traces are found, OVERRIDE the AI prediction!
        if camera_score > 0.60:
            results.at[idx, "predicted"] = 0
            fixed_count += 1

print(f"[Secondary Pipeline] Successfully overrode and fixed {fixed_count} False Positives!\n")
# ==========================================================


# --- Filter for original (untransformed) images in test set ---
originals = results[results["attack_type_str"] == "none"].copy()
print(f"Original (untransformed) images in test set: {len(originals)}")

# --- Correctly classified ---
correct = originals[originals["actual"] == originals["predicted"]]
print(f"Correctly classified originals: {len(correct)}")

# --- Incorrectly classified ---
incorrect = originals[originals["actual"] != originals["predicted"]]
print(f"Incorrectly classified originals (NOT stored): {len(incorrect)}")

# --- Store correctly classified originals ---
real_stored = 0
ai_stored = 0
skipped = 0

for _, row in correct.iterrows():
    src_path = str(row["original_path"])

    if not os.path.exists(src_path):
        skipped += 1
        continue

    filename = os.path.basename(src_path)

    if row["actual"] == 0 and row["predicted"] == 0:
        dst = os.path.join(CORRECT_REAL_DIR, filename)
        shutil.copy2(src_path, dst)
        real_stored += 1

    elif row["actual"] == 1 and row["predicted"] == 1:
        dst = os.path.join(CORRECT_AI_DIR, filename)
        shutil.copy2(src_path, dst)
        ai_stored += 1

print(f"\nStored correctly classified originals:")
print(f"  Real images (label=0, vote=0) : {real_stored} --> {CORRECT_REAL_DIR}")
print(f"  AI images   (label=1, vote=1) : {ai_stored}  --> {CORRECT_AI_DIR}")
print(f"  Skipped (file not found)      : {skipped}")
print(f"  Total stored                  : {real_stored + ai_stored}")

# --- Summary ---
print("\n" + "=" * 50)
print("PIPELINE COMPLETE (v2)")
print("=" * 50)
print(f"Real CSV     : {REAL_CSV}")
print(f"AI CSV       : {AI_CSV}")
print(f"Combined CSV : {COMBINED_CSV}")
print(f"Errors CSV   : {os.path.join(PROJECT_DIR, 'fp_fn_images_v2.csv')}")
print(f"Model        : {os.path.join(MODEL_DIR, 'ensemble_v2.joblib')}")
print(f"Real originals stored in : {CORRECT_REAL_DIR}")
print(f"AI originals stored in   : {CORRECT_AI_DIR}")


[Secondary Pipeline] Found 210 False Positives. Routing to Tier 2 Deep Forensics...
[Secondary Pipeline] Successfully overrode and fixed 174 False Positives!

Original (untransformed) images in test set: 116
Correctly classified originals: 112
Incorrectly classified originals (NOT stored): 4

Stored correctly classified originals:
  Real images (label=0, vote=0) : 54 --> /kaggle/working/correctly_classified_v2/real
  AI images   (label=1, vote=1) : 58  --> /kaggle/working/correctly_classified_v2/ai
  Skipped (file not found)      : 0
  Total stored                  : 112

PIPELINE COMPLETE (v2)
Real CSV     : /kaggle/working/real_detector_dataset_v2.csv
AI CSV       : /kaggle/working/ai_detector_dataset_v2.csv
Combined CSV : /kaggle/working/combined_detector_dataset_v2.csv
Errors CSV   : /kaggle/working/fp_fn_images_v2.csv
Model        : /kaggle/working/models_v2/ensemble_v2.joblib
Real originals stored in : /kaggle/working/correctly_classified_v2/real
AI originals stored in   : /kagg

In [48]:
import os

for f in os.listdir("/kaggle/working"):
    print(f)

ai_detector_dataset_v2.csv
fp_fn_images_v2.csv
png_dataset
models_v2
.virtual_documents
real_detector_dataset_v2.csv
correctly_classified_v2
image_metadata_v2.csv
combined_detector_dataset_v2.csv


## Pipeline 3: Advanced Transformations
This cell defines the advanced adversarial techniques like Motion Blur, Chromatic Aberration, Poisson Noise, WebP compression, etc.

In [49]:
import numpy as np
import cv2
from PIL import Image
from io import BytesIO

def motion_blur(image, kernel_size=15, angle=45):
    """Simulates camera shake (motion blur)."""
    img_arr = np.array(image)
    M = cv2.getRotationMatrix2D((kernel_size / 2, kernel_size / 2), angle, 1)
    motion_blur_kernel = np.diag(np.ones(kernel_size))
    motion_blur_kernel = cv2.warpAffine(motion_blur_kernel, M, (kernel_size, kernel_size))
    motion_blur_kernel = motion_blur_kernel / kernel_size
    blurred = cv2.filter2D(img_arr, -1, motion_blur_kernel)
    return Image.fromarray(blurred)

def chromatic_aberration(image, shift=3):
    """Simulates lens color fringing by shifting RGB channels."""
    img_arr = np.array(image)
    r, g, b = cv2.split(img_arr)
    
    rows, cols = r.shape
    M_right = np.float32([[1, 0, shift], [0, 1, 0]])
    M_left = np.float32([[1, 0, -shift], [0, 1, 0]])
    
    r_shifted = cv2.warpAffine(r, M_right, (cols, rows))
    b_shifted = cv2.warpAffine(b, M_left, (cols, rows))
    
    merged = cv2.merge((r_shifted, g, b_shifted))
    return Image.fromarray(merged)

def poisson_noise(image):
    """Simulates low-light sensor noise (signal-dependent)."""
    img_arr = np.array(image) / 255.0
    noisy = np.random.poisson(img_arr * 255.0) / 255.0
    noisy = np.clip(noisy, 0, 1) * 255.0
    return Image.fromarray(noisy.astype(np.uint8))

def webp_compression(image, quality=50):
    """Simulates modern web compression artifacts."""
    buffer = BytesIO()
    image.save(buffer, format="WEBP", quality=quality)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")

def cutout_simulation(image, size=50):
    """Simulates localized cutout or missing regions (forgery)."""
    img_arr = np.array(image)
    h, w, _ = img_arr.shape
    if h > size and w > size:
        y = np.random.randint(0, h - size)
        x = np.random.randint(0, w - size)
        img_arr[y:y+size, x:x+size] = 0
    return Image.fromarray(img_arr)

def mixed_degradation(image):
    """Applies a combination of degradations: Blur + Noise + JPEG."""
    img = image.copy()
    img = motion_blur(img, kernel_size=9, angle=15)
    img = poisson_noise(img)
    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=60)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")

advanced_transformations = {
    "none": lambda x: x,
    "motion_blur_15": lambda x: motion_blur(x, 15, 45),
    "chromatic_aberration_3": lambda x: chromatic_aberration(x, 3),
    "poisson_noise": lambda x: poisson_noise(x),
    "webp_50": lambda x: webp_compression(x, 50),
    "cutout_50": lambda x: cutout_simulation(x, 50),
    "mixed_degradation": lambda x: mixed_degradation(x)
}


## Pipeline 3: Automated FN/FP Deep Forensics
This cell automatically extracts the incorrectly classified samples (FN and FP) from Pipeline 2, passes them through **ALL** advanced techniques sequentially, and evaluates them.

In [52]:
import os
import pandas as pd
from pathlib import Path
from PIL import Image

# Import from the previous pipeline
# from all_detectors import run_all_detectors
# from advanced_transformations import advanced_transformations

def run_automated_pipeline3(csv_path="/kaggle/working/fp_fn_images_v2.csv", max_images=10):
    print("==================================================")
    print("  Automated Pipeline 3: Deep Forensic Analysis")
    print("  (Running ALL Advanced Transforms on FN & FP)")
    print("==================================================\n")

    if not os.path.exists(csv_path):
        print(f"Error: {csv_path} not found.")
        print("Please ensure Pipeline 2 has completely finished evaluating the test set.")
        return

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Failed to read CSV: {e}")
        return

    if df.empty:
        print("No errors found in the CSV. Pipeline 3 has nothing to process!")
        return

    print(f"Loaded {len(df)} False Positives/Negatives. Processing up to {max_images} images...\n")
    
    transform_keys = list(advanced_transformations.keys())
    if "none" in transform_keys:
        transform_keys.remove("none") # Don't just re-test the original here

    results_list = []

    # Process images
    for idx, row in df.head(max_images).iterrows():
        img_path = ""
        for col in ["original_path", "image_path", "path", "image_id", "file_path"]:
            if col in row and pd.notna(row[col]) and str(row[col]).strip() != "":
                img_path = str(row[col])
                break
                
        actual = row.get("actual", "Unknown")
        predicted = row.get("predicted", "Unknown")
        error_type = row.get("error_type", "Unknown")

        if not img_path or img_path == "nan":
            print(f"[{error_type}] Invalid or missing image path in CSV.")
            continue

        path = Path(img_path)
        if not path.is_file():
            print(f"[{error_type}] Image missing or is a directory: {img_path}")
            continue

        try:
            original_image = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"[{error_type}] Failed to load {img_path}: {e}")
            continue

        print(f"\n--- Routing {error_type} (Actual: {actual}, Pred: {predicted}) ---")
        print(f"File: {path.name}")
        
        # APPLY ALL ADVANCED TRANSFORMATIONS
        print(f"  -> Applying all transformations in batch...")
        try:
            trans_subdict = {k: advanced_transformations[k] for k in transform_keys}
            attack_names, final_scores = run_all_detectors_batched(original_image, trans_subdict)
            
            for i, trans_key in enumerate(attack_names):
                res_dict = {
                    "image_id": row.get("image_id", path.name),
                    "error_type": error_type,
                    "applied_transformation": trans_key
                }
                res_dict.update(final_scores[i])
                results_list.append(res_dict)
            print("  Finished all transformations for this image.")
        except Exception as e:
            print(f"     Error during batch pipeline execution: {e}")

    # Save output
    if results_list:
        out_df = pd.DataFrame(results_list)
        out_dir = "/kaggle/working"
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, "pipeline3_attack_results.csv")
        out_df.to_csv(out_path, index=False)
        print(f"\nPipeline 3 complete! Forensic features for all attacks saved to: {out_path}")
    else:
        print("\nNo features extracted.")

if __name__ == "__main__":
    run_automated_pipeline3()



  Automated Pipeline 3: Deep Forensic Analysis
  (Running ALL Advanced Transforms on FN & FP)

Loaded 297 False Positives/Negatives. Processing up to 10 images...


--- Routing False Positive (Actual: 0, Pred: 1) ---
File: (21).jpg
  -> Applying all transformations in batch...
  Finished all transformations for this image.

--- Routing False Positive (Actual: 0, Pred: 1) ---
File: (21).jpg
  -> Applying all transformations in batch...
  Finished all transformations for this image.

--- Routing False Positive (Actual: 0, Pred: 1) ---
File: (21).jpg
  -> Applying all transformations in batch...
  Finished all transformations for this image.

--- Routing False Positive (Actual: 0, Pred: 1) ---
File: (21).jpg
  -> Applying all transformations in batch...
  Finished all transformations for this image.

--- Routing False Positive (Actual: 0, Pred: 1) ---
File: (21).jpg
  -> Applying all transformations in batch...
  Finished all transformations for this image.

--- Routing False Positive (Ac

## Real-Time Interactive Test Case
Finally, use this cell to interactively test any image in real-time. Just type the path, pick an advanced transformation, and see the results instantly!

In [1]:
import os
import time
import json
import joblib
import pandas as pd
from pathlib import Path
from PIL import Image

# 1. Load your trained model and features
MODEL_DIR = "/kaggle/working/models_v2" 
try:
    ensemble = joblib.load(os.path.join(MODEL_DIR, "ensemble_v2.joblib"))
    with open(os.path.join(MODEL_DIR, "features_v2.json"), "r") as f:
        model_features = json.load(f)
    print("Successfully loaded trained Ensemble model!")
except Exception as e:
    print(f"Warning: Could not load trained model ({e}). Did you finish training it?")
    ensemble = None

# 2. Combine both transformation dictionaries into one massive list
all_transformations = {}
all_transformations.update(transformations)          # The 38 basic edits
all_transformations.update(advanced_transformations) # The 7 advanced forensics

print(f"Total unique transformations combined: {len(all_transformations)}")

def main():
    print("\n==================================================")
    print("  Bulk Forensic Analysis (All Transformations)")
    print("==================================================\n")

    while True:
        image_path = input("Enter the path to your image (or 'q' to quit): ").strip()
        if image_path.lower() == 'q':
            break

        path = Path(image_path)
        if not path.exists() or path.is_dir():
            print(f"Error: Invalid image path.\n")
            continue

        try:
            original_image = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"Failed to open image: {e}\n")
            continue

        if ensemble is None:
            print("Cannot run predictions without the model loaded.")
            continue

        print(f"\nProcessing {path.name}...")
        print(f"Applying all {len(all_transformations)} transformations and extracting features...")
        start_time = time.time()
        
        # 3. Run all transformations automatically using the batched pipeline
        try:
            attack_names, batch_scores = run_all_detectors_batched(original_image, all_transformations)
        except Exception as e:
            print(f"Error during feature extraction: {e}\n")
            continue
            
        print("Feature extraction complete! Running AI vs Real prediction...")
        
        # 4. Format features for the model
        results_df = pd.DataFrame(batch_scores)
        for col in model_features:
            if col not in results_df.columns:
                results_df[col] = 0 # Safety fallback
                
        # 5. Predict on all 44 rows at once!
        X_test = results_df[model_features]
        predictions = ensemble.predict(X_test)
        probabilities = ensemble.predict_proba(X_test)[:, 1]
        
        # 6. Build the final output CSV
        output_data = []
        for i, attack_name in enumerate(attack_names):
            row_data = {
                "image_filename": path.name,
                "transformation_applied": attack_name,
                "ai_probability": f"{probabilities[i]*100:.2f}%",
                "FINAL_VERDICT": "AI" if predictions[i] == 1 else "Real"
            }
            # We append the raw detector scores at the end of the row
            row_data.update(batch_scores[i])
            output_data.append(row_data)
            
        final_df = pd.DataFrame(output_data)
        
        # 7. Save to CSV in Kaggle working directory
        output_csv = f"/kaggle/working/interactive_results_{path.stem}.csv"
        final_df.to_csv(output_csv, index=False)
        
        end_time = time.time()
        
        # 8. Print Summary
        ai_count = (final_df['FINAL_VERDICT'] == 'AI').sum()
        real_count = (final_df['FINAL_VERDICT'] == 'Real').sum()
        
        print(f"\n==================================================")
        print(f"DONE! Execution time: {end_time - start_time:.2f} seconds")
        print(f"Summary for {path.name}:")
        print(f"  - Detected as AI   : {ai_count} times")
        print(f"  - Detected as Real : {real_count} times")
        print(f"\nDetailed results saved to: {output_csv}")
        print("==================================================\n")

if __name__ == "__main__":
    main()


NameError: name 'transformations' is not defined

In [57]:
import os
import json
import time
from pathlib import Path
from PIL import Image


def main():
    print("==================================================")
    print("  Interactive Real-Time Test Case Execution")
    print("  (Pipeline 3 + Advanced Transformations)")
    print("==================================================\n")

    while True:
        image_path = input("Enter the path to your image (or 'q' to quit): ").strip()
        if image_path.lower() == 'q':
            break

        # Remove surrounding quotes if dragged-and-dropped in terminal
        if image_path.startswith('"') and image_path.endswith('"'):
            image_path = image_path[1:-1]

        path = Path(image_path)
        if not path.exists():
            print(f"Error: Could not find image at {path}\n")
            continue

        try:
            print("Loading image...")
            original_image = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"Failed to open image: {e}\n")
            continue

        print("\nAvailable Advanced Transformations:")
        keys = list(advanced_transformations.keys())
        for i, key in enumerate(keys):
            print(f"  {i}: {key}")
        
        trans_idx = input(f"\nSelect a transformation to apply (0-{len(keys)-1}) [Default: 0]: ").strip()
        if trans_idx == "":
            trans_idx = 0
        try:
            trans_idx = int(trans_idx)
            selected_trans = keys[trans_idx]
        except (ValueError, IndexError):
            print("Invalid selection, defaulting to 0 (none).")
            selected_trans = "none"

        print(f"\nApplying transformation: {selected_trans}...")
        try:
            test_image = advanced_transformations[selected_trans](original_image)
        except Exception as e:
            print(f"Error applying transformation: {e}\n")
            continue

        print("\nExecuting Pipeline 3: Running 6 improved detectors...")
        start_time = time.time()
        
        try:
            # Tell the batched function to process the image with a dummy "none" transformation
            _, batch_scores = run_all_detectors_batched(test_image, {"none": lambda x: x})
            results = batch_scores[0] # Grab the results from the first (and only) image
        except Exception as e:
            print(f"Error during detector execution: {e}\n")
            continue
        end_time = time.time()

        print(f"\n--- Results (Execution time: {end_time - start_time:.2f}s) ---")
        print(json.dumps(results, indent=4))
        print("--------------------------------------------------\n")

if __name__ == "__main__":
    main()


  Interactive Real-Time Test Case Execution
  (Pipeline 3 + Advanced Transformations)



Enter the path to your image (or 'q' to quit):  /kaggle/input/datasets/ishu15m/test-image1/ChatGPT Image Jun 17 2026 02_11_06 PM.png


Loading image...

Available Advanced Transformations:
  0: none
  1: motion_blur_15
  2: chromatic_aberration_3
  3: poisson_noise
  4: webp_50
  5: cutout_50
  6: mixed_degradation



Select a transformation to apply (0-6) [Default: 0]:  0



Applying transformation: none...

Executing Pipeline 3: Running 6 improved detectors...

--- Results (Execution time: 0.75s) ---
{
    "siglip_ai_prob": 0.9999827146530151,
    "siglip_confidence": 0.9999827146530151,
    "vit_ai_prob": 0.011362523771822453,
    "vit_confidence": 0.9886375069618225,
    "fft_low_energy": 9.194720268249512,
    "fft_mid_energy": 7.787811279296875,
    "fft_high_energy": 7.099366664886475,
    "fft_high_freq_ratio": 0.2846702039241791,
    "fft_entropy": 14.257476806640625,
    "fft_mid_to_high_ratio": 1.0969726803527076,
    "ela_mean_q95": 0.9760116934776306,
    "ela_std_q95": 1.0755420923233032,
    "ela_max_q95": 34.0,
    "ela_mean_q75": 1.5021767616271973,
    "ela_std_q75": 1.792479157447815,
    "ela_skew": 3.7495710849761963,
    "ela_kurtosis": 42.51000213623047,
    "noise_std_gauss": 2.712294340133667,
    "noise_mean_gauss": 1.1104350090026855,
    "noise_std_median": 3.341221332550049,
    "noise_laplacian_var": 91.57648468017578,
    "no

Enter the path to your image (or 'q' to quit):  q
